In [15]:
import warnings
import sys

from pathlib import Path
from tqdm.auto import tqdm

import networkx as nx
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns


PROJECT_ROOT = Path("/home/frantsuaza/Documents/Science/SportLab/collusion_judges")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from collusion_judges.config import LONG_DF_PATH
from collusion_judges.dataset.make_graph import JudgeGraphBuilder

%load_ext autoreload
%autoreload 2

warnings.filterwarnings("ignore")

%config InlineBackend.figure_format = "retina"

plt.rcParams["figure.figsize"] = (8, 5)
plt.rcParams["font.size"] = 12
plt.rcParams["savefig.format"] = "pdf"

sns.set_style("darkgrid")

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [8]:
long_df = pd.read_excel(LONG_DF_PATH)

print(long_df.columns)

long_df.sample(5)

Index(['athlete_name', 'athlete_country', 'category', 'type',
       'program_numeric', 'year', 'stage', 'program', 'event', 'event_country',
       'judge_country', 'judge_GOE', 'judge_tech_score',
       'judge_component_score', 'judge_tech_place', 'judge_component_place',
       'judge_name', 'region', 'cpi_score', 'cpi_rank', 'rating_place',
       'rating_points', 'same_country', 'home_advantage', 'coef',
       'judge_component_score_norm'],
      dtype='object')


,athlete_name,athlete_country,category,type,program_numeric,year,stage,program,event,event_country,...,judge_name,region,cpi_score,cpi_rank,rating_place,rating_points,same_country,home_advantage,coef,judge_component_score_norm
3148,Hyungyeom Kim,KOR,men,1,1,2024,GP Cup of China,FreeSkate,2024_GP_Cup_of_China,CHN,...,Sookyung LEE,AP,64.0,30.0,26.0,984.0,1,0,3.33,22.252252
10447,Kaiya Ruiter,CAN,women,0,1,2023,GP Skate Canada,FreeSkate,2023_GP_Skate_Canada,CAN,...,Yoko KUNO,AP,73.0,16.0,57.0,464.0,0,1,2.67,20.501873
3726,Smirnova / Siianytsia,USA,pairs,2,1,2022,GP Espoo,FreeSkate,2022_GP_Espoo,FIN,...,Miriam PALANGE,WE/EU,56.0,41.0,34.0,695.0,0,0,2.67,21.003745
12600,Alysa Liu,USA,women,0,0,2024,GP Skate Canada Intl,ShortProgram,2024_GP_Skate_Canada_Intl,CAN,...,Donatella LEONELLI,WE/EU,81.0,5.0,NaN,NaN,0,0,1.33,23.007519
6311,Lukas Britschgi,CHE,men,1,0,2024,GP Finlandia Trophy,ShortProgram,2024_GP_Finlandia_Trophy,FIN,...,Anu NIINIRANTA,WE/EU,88.0,2.0,6.0,1895.0,0,0,1.67,24.000000


In [16]:
builder = JudgeGraphBuilder(
    long_df=long_df,
    score_cols={
        "components": "judge_component_score_norm",
        "technique": "judge_GOE",
    },
    corr_types=(
        "pearson",
        "spearman",
        "kendall",
    ),
)

G = builder.build_graph()

In [17]:
example_judge = next(iter(G.nodes))
node = G.nodes[example_judge]

print("Judge:", example_judge)
print("Country:", node["country"])

print("\nComponents, Pearson:")
print(node["components_pearson_mean"][:5])

print("\nTechnique, Pearson:")
print(node["technique_pearson_mean"][:5])

Judge: Lorrie PARKER
Country: USA

Components, Pearson:
[np.float64(0.8126461994572663), np.float64(0.9069969187815639), np.float64(0.8034388906251678), np.float64(0.8630939237777651), np.float64(0.7768301948614009)]

Technique, Pearson:
[np.float64(0.9484565801323622), np.float64(0.9162537100259789), np.float64(0.7982853698785122), np.float64(0.963145570206866), np.float64(0.9140832028624772)]


In [18]:
judge_1, judge_2 = next(iter(G.edges))
edge = G[judge_1][judge_2]

print("judge_a:", edge["judge_a"])
print("judge_b:", edge["judge_b"])

print("\nComponents, Pearson:")
print(edge["components_pearson_r"][:5])

print("\nTechnique, Pearson:")
print(edge["technique_pearson_r"][:5])

print("\nRanks for judge_a:")
print(edge["components_pearson_rank_a"][:5])

print("\nRanks for judge_b:")
print(edge["components_pearson_rank_b"][:5])

judge_a: Albert ZAYDMAN
judge_b: Lorrie PARKER

Components, Pearson:
[np.float64(0.8593901505294181), np.float64(0.9175862121083291)]

Technique, Pearson:
[np.float64(0.7824501401722791), np.float64(0.9150560997157349)]

Ranks for judge_a:
[np.float64(6.0), np.float64(6.0)]

Ranks for judge_b:
[np.float64(5.0), np.float64(4.0)]


# **Old code with classes**

## Old but no that old

In [ ]:
# collusion_judges/dataset/make_graph.py
from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path
from typing import Iterable, Optional

import numpy as np
import pandas as pd
import networkx as nx

from itertools import combinations
from typing import Sequence


from collusion_judges.config import LONG_DF_PATH


class JudgeGraphBuilder:
    """
    Build a judge graph from long-format scores.

    Nodes store judge-level statistics for each judging act.
    Edges store pairwise correlations and pair ranks.
    """

    def __init__(
        self,
        long_df: pd.DataFrame,
        score_col: str = "judge_component_score_norm",
        corr_types: Sequence[str] = ("pearson",),
    ) -> None:
        self.df = long_df.copy()
        self.score_col = score_col
        self.corr_types = tuple(corr_types)

        self.athlete_col = "athlete_name"
        self.judge_col = "judge_name"
        self.judge_country_col = "judge_country"

        self.event_descr = [
            "year",
            "stage",
            "category",
            "program",
        ]

        self.judge_statistics = [
            "mean",
            "median",
            "min",
            "max",
            "std",
            "mad",
            "iqr",
        ]

        self.events = tuple(
            self.df[self.event_descr]
            .drop_duplicates()
            .itertuples(index=False, name=None)
        )

        self.G = nx.Graph()
        self._graph_is_filled = False

        self._initiate_graph()

    # ------------------------------------------------------------------
    # Basic helpers
    # ------------------------------------------------------------------

    @staticmethod
    def _ordered_pair(
        judge_1: str,
        judge_2: str,
    ) -> tuple[str, str]:
        """
        Return judges in deterministic lexicographic order.

        The first judge is always judge_a; the second is judge_b.
        """
        return tuple(
            sorted(
                (judge_1, judge_2),
                key=lambda name: (name.casefold(), name),
            )
        )

    def _get_judges(self, df: pd.DataFrame) -> list[str]:
        """Return judges in deterministic lexicographic order."""
        judges = (
            df[self.judge_col]
            .dropna()
            .unique()
            .tolist()
        )

        return sorted(
            judges,
            key=lambda name: (name.casefold(), name),
        )

    # ------------------------------------------------------------------
    # Graph initialization
    # ------------------------------------------------------------------

    def _get_node_attributes(self) -> dict[str, list]:
        """Create empty attributes for one node."""
        attributes = {
            attr: []
            for attr in self.event_descr + ["n_athletes"]
        }

        for corr in self.corr_types:
            for statistic in self.judge_statistics:
                attributes[f"{corr}_{statistic}"] = []

            attributes[f"{corr}_rank_by_mean"] = []
            attributes[f"{corr}_rank_by_median"] = []

            if corr == "pearson":
                attributes["pearson_fisher_mean"] = []
                attributes["pearson_rank_by_fisher_mean"] = []

        return attributes

    def _get_edge_attributes(
        self,
        judge_a: str,
        judge_b: str,
    ) -> dict:
        """Create empty attributes for one edge."""
        attributes = {
            "judge_a": judge_a,
            "judge_b": judge_b,
        }

        attributes.update(
            {
                attr: []
                for attr in self.event_descr + ["n_athletes"]
            }
        )

        for corr in self.corr_types:
            attributes[f"{corr}_r"] = []
            attributes[f"{corr}_rank_a"] = []
            attributes[f"{corr}_rank_b"] = []

        return attributes

    def _initiate_nodes(self) -> None:
        """Add all judges and initialize node attributes."""
        judge_info = (
            self.df[
                [
                    self.judge_col,
                    self.judge_country_col,
                ]
            ]
            .dropna(subset=[self.judge_col])
            .drop_duplicates(subset=self.judge_col)
            .set_index(self.judge_col)[self.judge_country_col]
        )

        nodes = []

        for judge, country in judge_info.items():
            attributes = self._get_node_attributes()
            attributes["country"] = country

            nodes.append((judge, attributes))

        self.G.add_nodes_from(nodes)

    def _initiate_edges(self) -> None:
        """Add every pair of judges that appears in a common panel."""
        all_pairs: set[tuple[str, str]] = set()

        event_groups = self.df.groupby(
            self.event_descr,
            sort=False,
            dropna=False,
        )

        for _, event_df in event_groups:
            judges = self._get_judges(event_df)

            event_pairs = {
                self._ordered_pair(judge_1, judge_2)
                for judge_1, judge_2 in combinations(judges, 2)
            }

            all_pairs.update(event_pairs)

        edges = [
            (
                judge_a,
                judge_b,
                self._get_edge_attributes(judge_a, judge_b),
            )
            for judge_a, judge_b in all_pairs
        ]

        self.G.add_edges_from(edges)

    def _initiate_graph(self) -> None:
        """Create all nodes, edges, and empty attributes."""
        self._initiate_nodes()
        self._initiate_edges()

    # ------------------------------------------------------------------
    # Event data
    # ------------------------------------------------------------------

    def _get_score_matrix(
        self,
        event_df: pd.DataFrame,
    ) -> pd.DataFrame:
        """Return an athlete-by-judge score matrix."""
        score_matrix = (
            event_df.pivot_table(
                index=self.athlete_col,
                columns=self.judge_col,
                values=self.score_col,
                aggfunc="mean",
            )
            .dropna(axis=0, how="all")
            .dropna(axis=1, how="all")
        )

        ordered_judges = sorted(
            score_matrix.columns,
            key=lambda name: (name.casefold(), name),
        )

        return score_matrix.reindex(columns=ordered_judges)

    @staticmethod
    def _get_corr_matrix(
        score_matrix: pd.DataFrame,
        corr: str,
    ) -> pd.DataFrame:
        """
        Calculate a judge correlation matrix.

        The diagonal is replaced with NaN so that self-correlations
        do not affect statistics and ranks.
        """
        corr_matrix = score_matrix.corr(method=corr)
        np.fill_diagonal(corr_matrix.values, np.nan)

        return corr_matrix

    # ------------------------------------------------------------------
    # Primary node metrics
    # ------------------------------------------------------------------

    @staticmethod
    def _get_judge_statistics(
        corr_matrix: pd.DataFrame,
        corr: str,
    ) -> pd.DataFrame:
        """Calculate event-level correlation statistics for each judge."""
        medians = corr_matrix.median(axis=1)

        quartiles = corr_matrix.quantile(
            [0.25, 0.75],
            axis=1,
        ).T

        statistics = pd.DataFrame(
            {
                "mean": corr_matrix.mean(axis=1),
                "median": medians,
                "min": corr_matrix.min(axis=1),
                "max": corr_matrix.max(axis=1),
                "std": corr_matrix.std(axis=1, ddof=1),
                "mad": (
                    corr_matrix
                    .sub(medians, axis=0)
                    .abs()
                    .median(axis=1)
                ),
                "iqr": quartiles[0.75] - quartiles[0.25],
            },
            index=corr_matrix.index,
        )

        if corr == "pearson":
            clipped = corr_matrix.clip(
                lower=-1 + 1e-12,
                upper=1 - 1e-12,
            )

            fisher_z = np.arctanh(clipped)

            statistics["fisher_mean"] = np.tanh(
                fisher_z.mean(axis=1)
            )

        return statistics

    @staticmethod
    def _get_judge_ranks(
        statistics: pd.DataFrame,
    ) -> pd.DataFrame:
        """
        Rank judges by agreement with the panel.

        Rank 1 corresponds to the highest agreement.
        """
        ranks = pd.DataFrame(index=statistics.index)

        ranks["rank_by_mean"] = statistics["mean"].rank(
            ascending=False,
            method="min",
        )

        ranks["rank_by_median"] = statistics["median"].rank(
            ascending=False,
            method="min",
        )

        if "fisher_mean" in statistics.columns:
            ranks["rank_by_fisher_mean"] = statistics[
                "fisher_mean"
            ].rank(
                ascending=False,
                method="min",
            )

        return ranks

    # ------------------------------------------------------------------
    # Primary edge metrics
    # ------------------------------------------------------------------

    @staticmethod
    def _get_pair_rank_matrix(
        corr_matrix: pd.DataFrame,
    ) -> pd.DataFrame:
        """
        Rank every partner separately for each judge.

        Rank 1 corresponds to the highest pairwise correlation.
        """
        return corr_matrix.rank(
            axis=1,
            ascending=False,
            method="min",
            na_option="keep",
        )

    # ------------------------------------------------------------------
    # Collect event parameters
    # ------------------------------------------------------------------

    def _collect_event_parameters(
        self,
        event: tuple,
        event_df: pd.DataFrame,
    ) -> dict:
        """Collect all primary parameters for one judging act."""
        score_matrix = self._get_score_matrix(event_df)

        parameters = {
            **dict(zip(self.event_descr, event)),
            "judges": score_matrix.columns.tolist(),
            "n_athletes": score_matrix.shape[0],
            "correlations": {},
        }

        for corr in self.corr_types:
            corr_matrix = self._get_corr_matrix(
                score_matrix,
                corr,
            )

            statistics = self._get_judge_statistics(
                corr_matrix,
                corr,
            )

            judge_ranks = self._get_judge_ranks(statistics)
            pair_ranks = self._get_pair_rank_matrix(corr_matrix)

            parameters["correlations"][corr] = {
                "matrix": corr_matrix,
                "statistics": statistics,
                "judge_ranks": judge_ranks,
                "pair_ranks": pair_ranks,
            }

        return parameters

    # ------------------------------------------------------------------
    # Push data to nodes
    # ------------------------------------------------------------------

    def _push_event_to_nodes(
        self,
        parameters: dict,
    ) -> None:
        """Append one judging act's metrics to node attributes."""
        for judge in parameters["judges"]:
            node = self.G.nodes[judge]

            for attr in self.event_descr:
                node[attr].append(parameters[attr])

            node["n_athletes"].append(
                parameters["n_athletes"]
            )

            for corr, corr_data in parameters[
                "correlations"
            ].items():
                statistics = corr_data["statistics"].loc[judge]
                ranks = corr_data["judge_ranks"].loc[judge]

                for statistic in self.judge_statistics:
                    node[f"{corr}_{statistic}"].append(
                        statistics[statistic]
                    )

                node[f"{corr}_rank_by_mean"].append(
                    ranks["rank_by_mean"]
                )

                node[f"{corr}_rank_by_median"].append(
                    ranks["rank_by_median"]
                )

                if corr == "pearson":
                    node["pearson_fisher_mean"].append(
                        statistics["fisher_mean"]
                    )

                    node[
                        "pearson_rank_by_fisher_mean"
                    ].append(
                        ranks["rank_by_fisher_mean"]
                    )

    # ------------------------------------------------------------------
    # Push data to edges
    # ------------------------------------------------------------------

    def _push_event_to_edges(
        self,
        parameters: dict,
    ) -> None:
        """Append one judging act's metrics to edge attributes."""
        for judge_1, judge_2 in combinations(
            parameters["judges"],
            2,
        ):
            judge_a, judge_b = self._ordered_pair(
                judge_1,
                judge_2,
            )

            edge = self.G[judge_a][judge_b]

            for attr in self.event_descr:
                edge[attr].append(parameters[attr])

            edge["n_athletes"].append(
                parameters["n_athletes"]
            )

            for corr, corr_data in parameters[
                "correlations"
            ].items():
                edge[f"{corr}_r"].append(
                    corr_data["matrix"].loc[
                        judge_a,
                        judge_b,
                    ]
                )

                edge[f"{corr}_rank_a"].append(
                    corr_data["pair_ranks"].loc[
                        judge_a,
                        judge_b,
                    ]
                )

                edge[f"{corr}_rank_b"].append(
                    corr_data["pair_ranks"].loc[
                        judge_b,
                        judge_a,
                    ]
                )

    def _push_event_data(
        self,
        parameters: dict,
    ) -> None:
        """Push collected event parameters to nodes and edges."""
        self._push_event_to_nodes(parameters)
        self._push_event_to_edges(parameters)

    # ------------------------------------------------------------------
    # Public interface
    # ------------------------------------------------------------------

    def add_info_to_graph(self) -> nx.Graph:
        """Calculate and add primary metrics for all judging acts."""
        if self._graph_is_filled:
            return self.G

        event_groups = self.df.groupby(
            self.event_descr,
            sort=False,
            dropna=False,
        )

        for event, event_df in event_groups:
            parameters = self._collect_event_parameters(
                event,
                event_df,
            )

            self._push_event_data(parameters)

        self._graph_is_filled = True
        return self.G

    def build_graph(self) -> nx.Graph:
        """Return the initialized and populated graph."""
        return self.add_info_to_graph()


## Old old and old old old

In [ ]:
# collusion_judges/dataset/make_graph.py

from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path
from typing import Iterable, Optional

import numpy as np
import pandas as pd
import networkx as nx

from collusion_judges.config import LONG_DF_PATH


@dataclass
class GraphConfig:
    event_col: str = "event"
    stage_col: str = "stage"
    year_col: str = "year"
    category_col: str = "type"
    program_col: str = "program_numeric"
    athlete_col: str = "athlete_name"
    judge_col: str = "judge_name"
    judge_country_col: str = "judge_country"
    score_col: str = "judge_component_score_norm"
    corr_methods: tuple[str, ...] = ("pearson", "spearman", "kendall")
    min_common: int = 3


# Не хочу делать отдельный конфиг
class JudgeGraphBuilder2:
    def __init__(self, long_df:pd.DataFrame=None, score_col:str='judge_component_score_norm', corr_types:list =['pearson',] ):
        self.df = long_df
        self.events = None
        self.score_col = score_col
        self.corr_types = corr_types
        self.G = nx.Graph()
        self.event_desr = ['year', 'stage', 'category', 'program',]
        self.judge_descr = ['mean', 'median', 'min', 'max', 'std', 'MAD', 'IQR']
        # Может я что-то забыла ? (базовое описание корреляций у одного судьи в рамках одного акта судейства)
        # Не уверена нужно ли n_athletes
        self.edge_descr = ['A_rank', 'B_rank',] + self.corr_types
        self.initiate_graph()

        # judges = long_df

    def get_judges(self, df):
        return df['judge_name'].unique()

    def make_all_stata(self):
        self.all_statistics = [f'{corr}_{stata}' for corr in self.corr_types for stata in self.judge_descr]
        if 'pearson' in self.corr_types:
            self.all_statistics.append('fisher_mean')

    def initiate_nodes(self):
        judges = self.get_judges(self.df)
        self.G.add_nodes_from(judges)
        # Было бы круто, если бы можно было сделать вычисления более быстрыми
        # Может можно векторно присвоить сразу всем вершинам одинаковые значения
        # для каждого атрибута
        # self.G[judges][attr] = list()
        # Или может хотя бы сразу по всем параметрам

        for judge in judges:
            for attr in self.make_all_stata() + self.event_descr:
                self.G[judge][attr] = list()


    def initiate_edges(self, event_df):
        edge_params = {param : list()  param for self.edge_params + self.event_descr}
        event_judges = self.get_judges(event_df)
        edges_to_add = []
        for i in range(len(event_judges)):
            judge_A = event_judges[i]
            for j in range(1, len(event_judges)):
                judge_B = event_judges[j]
                if self.G.has_edge(judge_A, judge_B):
                    continue

                #self.make_edge(judge_A, judge_B)
                edges_to_add.append((judge_A, judge_B, edge_params))
        if len(edges_to_add) > 0:
            self.add_edges_from(edges_to_add)



    def initiate_graph(self):
        '''
        Initiate graph (make all nodes and edges)
        '''

        self.initiate_nodes()

        # Надо сохранить список/кортеж уникальных комбинаций/событий, где судили те или иные бригады судей. Это должен быть кортеж из кортежей.
        self.events = self.df[self.event_descr]].unique()

        for year, stage, category, program in events:
            condition = (df['year'] == year) &  (df['stage'] == stage) &  (df['category'] == category) &  (df['program'] == program)

            event_df = self.df[condition]
            #event_judges = self.get_judges(event_df)
            # add node между всеми судьями
            #self.initiate_edges(event_df)


    def get_corr_matrix(self, df, corr:str='pearson'):
        wide = df.pivot_table(
            index=self.athlete_col,
            columns=self.judge_col,
            values=self.score_col,
            aggfunc="mean",  # Я не уверена
        ).dropna(axis=1, how="all")

        return wide.corr(corr_type=corr)

    def get_statistics(self, corr_matrix, parameters, corr):
        # Для каждого судьи надо получить максимум/минимум/медиану/др.статистики его корреляций
        # Судей всегда 9, поэтому у каждого судьи есть по 8 корреляций. Надо смотреть максимум помимо корреляции с самим собой.
        # ВЫчисления должны быть векторными.

        parameters[corr]['mean_arr'] = mean_arr
        parameters[corr]['median_arr'] = median_arr
        parameters[corr]['min_arr'] = min_arr
        parameters[corr]['max_arr'] = max_arr
        parameters[corr]['std_arr'] = std_arr
        parameters[corr]['IQR'] = IQR
        # Может быть я забыла здесь про какие-то параметры
        if corr == 'pearson':
            parameters[corr]['fisher_mean_arr'] = fisher_mean_arr



    def get_ranks_for_judge(self, mean_arr, median_arr):
        rank_by_mean = mean_arr.argsort().argsort()
        rank_by_median = median_arr.argsort().argsort()
        return rank_by_mean, rank_by_median

    def get_ranks_for_pair(self, corr_matrix):
        # Хочу получить для каждого судьи посмотреть на ранги его корреляций
        # с другими судьями
        # Эта информация понадобится, чтобы добавить в ребро информацию о том,
        # насколько сильной была это корреляция для одного и для другого

        rank_matrix = np.values((9, 9), -1)
        for i in range(9):
            rank_matrix[i] = corr_matrix[i].argsort().argsort()

        return rank_matrix



    def get_ranks(self, corr_matrix, mean_arr, median_arr, parameters, corr):
        rank_by_mean, rank_by_median = self.get_ranks_for_judge(mean_arr, median_arr)
        parameters[corr]['rank_by_mean'] = rank_by_mean
        parameters[corr]['rank_by_median'] = rank_by_median

        rank_matrix = self.get_ranks_for_pair(corr_matrix)
        parameters[corr]['rank_matrix'] = rank_matrix

        return parameters

    def push_nodes(self, parameters, corr):
        for i, judge in enumerate(parameters['judges'] ):

            for stata in self.judge_descr:
                self.G[judge][f'{corr}_{stata}'].append(parameters[corr][f'{stata}_{arr}'][i])
            for param in self.event_descr:
                self.G[judge][param].append(parameters[param])
        return None

    def push_edges(self, parameters, corr):
        for i, judge_A in enumerate(parameters['judges']):
            for j in range(i, len(parameters['judges'])):
                judge_B = parameters['judges'][j]
                # Надо добавить первичные параметры для ребра.


    def push_data(self, parameters, corr):
        self.push_nodes(parameters, corr)
        self.push_edges(parameters, corr)
        pass

    def add_info_to_graph(self):
        for year, stage, category, program in events:
            parameters = dict()
            parameters['year'] = year
            parameters['stage'] = stage
            parameters['category'] = category
            parameters['program'] = program

            condition = (df['year'] == year) &  (df['stage'] == stage) &  (df['category'] == category) &  (df['program'] == program)

            event_df = self.df[condition]
            parameters['judges'] = self.get_judges(event_df)

            for corr in self.corr_methods:
                parameters[corr] = dict()

                judges = self.get_judges()
                corr_matrix = self.get_corr_matrix(event_df, corr)
                self.get_statistics(corr_matrix, parameters, corr)
                self.get_ranks(corr_matrix, mean_arr, median_arr, parameters, corr)
                self.push_data(parameters, corr)




class JudgeGraphBuilder:
    """
    Builds a judge graph.

    Nodes are judges.
    Edges are pairs of judges who judged at least one common event/category/program.
    Each edge stores event-level correlations and aggregate statistics.
    """

    def __init__(self, df: pd.DataFrame, config: Optional[GraphConfig] = None) -> None:
        self.df = df.copy()
        self.config = config or GraphConfig()
        self.G = nx.Graph()

    def build_graph(self) -> nx.Graph:
        """Build graph from long-format judge scores."""
        self._validate_columns()
        self._add_nodes()

        group_cols = [
            self.config.event_col,
            self.config.category_col,
            self.config.program_col,
        ]

        optional_context_cols = [
            self.config.year_col,
            self.config.stage_col,
        ]

        for keys, sub in self.df.groupby(group_cols, dropna=False):
            event, category, program = keys

            context = {
                "event": event,
                "category": category,
                "program_numeric": program,
            }

            for col in optional_context_cols:
                if col in sub.columns:
                    context[col] = sub[col].iloc[0]

            self._add_edges_for_group(sub, context)

        self._compute_edge_aggregates()
        self._compute_node_aggregates()

        return self.G

    def get_edge_df(self) -> pd.DataFrame:
        """Return one row per judge pair."""
        rows = []

        for judge_a, judge_b, data in self.G.edges(data=True):
            row = {
                "judge_a": judge_a,
                "judge_b": judge_b,
                "judge_a_country": self.G.nodes[judge_a].get("country"),
                "judge_b_country": self.G.nodes[judge_b].get("country"),
                "n_common": data.get("n_common", 0),
            }

            for method in self.config.corr_methods:
                for stat in ["mean", "median", "min", "max", "std"]:
                    row[f"{method}_{stat}"] = data.get(f"{method}_{stat}")

            rows.append(row)

        return pd.DataFrame(rows)

    def get_node_df(self) -> pd.DataFrame:
        """Return one row per judge."""
        rows = []

        for judge, data in self.G.nodes(data=True):
            row = {
                "judge_name": judge,
                "judge_country": data.get("country"),
                "degree": self.G.degree(judge),
                "n_common_acts": data.get("n_common_acts", 0),
            }

            for method in self.config.corr_methods:
                for stat in ["mean", "median", "min", "max", "std"]:
                    row[f"{method}_{stat}"] = data.get(f"{method}_{stat}")

            rows.append(row)

        return pd.DataFrame(rows)

    def get_pair_corrs_df(self) -> pd.DataFrame:
        """
        Return tidy event-level correlation dataset.

        One row = one pair of judges in one event/category/program.
        This is the dataset you need for distribution plots.
        """
        rows = []

        for judge_a, judge_b, data in self.G.edges(data=True):
            for record in data.get("records", []):
                rows.append(
                    {
                        "judge_a": judge_a,
                        "judge_b": judge_b,
                        **record,
                    }
                )

        return pd.DataFrame(rows)

    def _validate_columns(self) -> None:
        required = [
            self.config.event_col,
            self.config.category_col,
            self.config.program_col,
            self.config.athlete_col,
            self.config.judge_col,
            self.config.judge_country_col,
            self.config.score_col,
        ]

        missing = [col for col in required if col not in self.df.columns]
        if missing:
            raise ValueError(f"Missing required columns: {missing}")

    def _add_nodes(self) -> None:
        judges = (
            self.df[[self.config.judge_col, self.config.judge_country_col]]
            .dropna(subset=[self.config.judge_col])
            .drop_duplicates()
        )

        for _, row in judges.iterrows():
            self.G.add_node(
                row[self.config.judge_col],
                country=row[self.config.judge_country_col],
                correlations={method: [] for method in self.config.corr_methods},
                n_common_acts=0,
            )

    def _add_edges_for_group(self, sub: pd.DataFrame, context: dict) -> None:
        wide = sub.pivot_table(
            index=self.config.athlete_col,
            columns=self.config.judge_col,
            values=self.config.score_col,
            aggfunc="mean",
        ).dropna(axis=1, how="all")

        if wide.shape[1] < 2:
            return

        corr_mats = {
            method: wide.corr(method=method, min_periods=self.config.min_common)
            for method in self.config.corr_methods
        }

        judges = list(wide.columns)

        for i, judge_a in enumerate(judges):
            self.G.nodes[judge_a]["n_common_acts"] += 1

            for judge_b in judges[i + 1:]:
                record = dict(context)
                has_valid_corr = False

                for method, corr_mat in corr_mats.items():
                    r = corr_mat.loc[judge_a, judge_b]
                    record[f"{method}_r"] = float(r) if pd.notna(r) else np.nan
                    has_valid_corr = has_valid_corr or pd.notna(r)

                if not has_valid_corr:
                    continue

                if not self.G.has_edge(judge_a, judge_b):
                    self.G.add_edge(
                        judge_a,
                        judge_b,
                        records=[],
                        n_common=0,
                    )

                self.G[judge_a][judge_b]["records"].append(record)
                self.G[judge_a][judge_b]["n_common"] += 1

    def _compute_edge_aggregates(self) -> None:
        for judge_a, judge_b, data in self.G.edges(data=True):
            records = data.get("records", [])

            for method in self.config.corr_methods:
                values = np.array(
                    [r[f"{method}_r"] for r in records],
                    dtype=float,
                )
                values = values[~np.isnan(values)]

                stats = self._safe_stats(values)
                for stat, value in stats.items():
                    self.G[judge_a][judge_b][f"{method}_{stat}"] = value

    def _compute_node_aggregates(self) -> None:
        for judge in self.G.nodes:
            for method in self.config.corr_methods:
                values = []

                for neighbor in self.G.neighbors(judge):
                    edge_values = [
                        r[f"{method}_r"]
                        for r in self.G[judge][neighbor].get("records", [])
                    ]
                    values.extend(edge_values)

                values = np.array(values, dtype=float)
                values = values[~np.isnan(values)]

                stats = self._safe_stats(values)
                for stat, value in stats.items():
                    self.G.nodes[judge][f"{method}_{stat}"] = value

    @staticmethod
    def _safe_stats(values: np.ndarray) -> dict[str, Optional[float]]:
        if len(values) == 0:
            return {
                "mean": None,
                "median": None,
                "min": None,
                "max": None,
                "std": None,
            }

        return {
            "mean": float(np.mean(values)),
            "median": float(np.median(values)),
            "min": float(np.min(values)),
            "max": float(np.max(values)),
            "std": float(np.std(values, ddof=1)) if len(values) > 1 else 0.0,
        }


def build_judge_graph(
    long_df: pd.DataFrame=None,
    score_col: str = "judge_component_score_norm",
    corr_methods: tuple[str, ...] = ("pearson", "spearman", "kendall"),
    min_common: int = 3,
) -> tuple[nx.Graph, pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """
    Convenience function for notebooks.

    Returns:
    - graph
    - edge_df
    - node_df
    - pair_corrs_df
    """
    config = GraphConfig(
        score_col=score_col,
        corr_methods=corr_methods,
        min_common=min_common,
    )
    if long_df == None:
        long_df = pd.read_excel(LONG_DF_PATH)

    builder = JudgeGraphBuilder(long_df, config=config)
    G = builder.build_graph()

    return (
        G,
        builder.get_edge_df(),
        builder.get_node_df(),
        builder.get_pair_corrs_df(),
    )
